python -m venv venv
venv\Scripts\activate
pip install numpy matplotlib tensorflow scikit-learn
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
pip install notebook
python -m notebook

In [21]:
import numpy as np

In [22]:
class ART1:
    def __init__(self, vigilance=0.8):
        self.vigilance = vigilance
        self.weights = []

    def _choice(self, x, w):
        # Choice function
        return np.sum(np.minimum(x, w)) / (0.0001 + np.sum(w))

    def _match(self, x, w):
        # Match function
        return np.sum(np.minimum(x, w)) / (0.0001 + np.sum(x))

    def train(self, data):
        for x in data:
            if len(self.weights) == 0:
                self.weights.append(x.copy())
                continue

            best_j = -1
            best_choice = -1

            for j, w in enumerate(self.weights):
                choice_val = self._choice(x, w)
                if choice_val > best_choice:
                    best_choice = choice_val
                    best_j = j

            if self._match(x, self.weights[best_j]) >= self.vigilance:
                # Update category
                self.weights[best_j] = np.minimum(x, self.weights[best_j])
            else:
                # Create new category
                self.weights.append(x.copy())

    def predict(self, x):
        scores = [self._choice(x, w) for w in self.weights]
        return np.argmax(scores)

In [23]:
patterns = np.array([
    [1, 0, 1, 0],
    [1, 0, 1, 1],
    [0, 1, 0, 1],
    [0, 1, 0, 0],
])

print("Input Patterns:")
for i, p in enumerate(patterns):
    print(f"Pattern {i}: {p}")

Input Patterns:
Pattern 0: [1 0 1 0]
Pattern 1: [1 0 1 1]
Pattern 2: [0 1 0 1]
Pattern 3: [0 1 0 0]


In [24]:
art = ART1(vigilance=0.75)
art.train(patterns)

print("Vigilance parameter:", art.vigilance)
print("Number of categories created:", len(art.weights))

Vigilance parameter: 0.75
Number of categories created: 3


In [25]:
print("\nCategory weights:")
for j, w in enumerate(art.weights):
    print(f"Category {j}: {w}")


Category weights:
Category 0: [1 0 1 0]
Category 1: [1 0 1 1]
Category 2: [0 1 0 0]


In [26]:
print("\nCategory assignments:")
for i, p in enumerate(patterns):
    category = art.predict(p)
    print(f"Pattern {i} {p} -> Category {category}")


Category assignments:
Pattern 0 [1 0 1 0] -> Category 0
Pattern 1 [1 0 1 1] -> Category 1
Pattern 2 [0 1 0 1] -> Category 2
Pattern 3 [0 1 0 0] -> Category 2


In [27]:
new_pattern = np.array([1, 0, 0, 0])
predicted_category = art.predict(new_pattern)

print(f"New pattern {new_pattern} -> Category {predicted_category}")

New pattern [1 0 0 0] -> Category 0
